# ACTIVIDAD 5: Modelo final

**Equipo 61**

Alumnos:

Gustavo Adolfo Morales García A00828432

Alejandro Jesús Mondragón Jiménez A01795837

Sebastián Ezequiel Coronado Rivera A01212824


### Objetivo

El objetivo de esta etapa consiste en desarrollar y evaluar modelos de ensamble capaces de mejorar el desempeño predictivo obtenido en las fases anteriores del proyecto. Para ello, se utilizarán los modelos individuales con mejor rendimiento como componentes base, buscando aprovechar la complementariedad de sus predicciones y reducir errores asociados a arquitecturas particulares.

La variable objetivo seleccionada para esta fase es LogMass, correspondiente al logaritmo de la masa estelar de una galaxia. Esta propiedad constituye una de las características físicas más relevantes en estudios de evolución galáctica, por lo que representa un caso de estudio adecuado para evaluar la capacidad de los modelos de visión computacional para inferir información física a partir de observaciones astronómicas.

Adicionalmente, se incorporan nuevos modelos entrenados utilizando imágenes obtenidas mediante un criterio de escalamiento basado en el radio de Petrosian. Este enfoque busca reducir la presencia de objetos cercanos no asociados a la galaxia objetivo, permitiendo que una mayor proporción de los píxeles disponibles contenga información relevante para el proceso de aprendizaje.

En esta etapa se consideran tanto modelos individuales como diferentes estrategias de ensamble. Se implementan enfoques homogéneos y heterogéneos, incluyendo métodos basados en promedio simple, promedio ponderado y técnicas de stacking. Posteriormente, los resultados son comparados utilizando métricas de regresión y tiempos de entrenamiento, con el objetivo de seleccionar el modelo final que ofrezca el mejor equilibrio entre desempeño predictivo y costo computacional.

In [19]:
# Importación de librerías necesarias
import os
import time
import numpy as np

from pathlib import Path
import pandas as pd  # Manipulación y análisis de datos
import numpy as np  # Operaciones numéricas y arrays
import matplotlib.pyplot as plt  # Visualización básica
import seaborn as sns  # Visualización estadística avanzada
from scipy import stats  # Funciones estadísticas
from scipy.stats import skew, kurtosis  # Métricas de distribución
import warnings  # Manejo de advertencias
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Configuración de estilos y opciones
warnings.filterwarnings('ignore')  # Ignorar advertencias para limpieza visual
sns.set_style('whitegrid')  # Estilo de gráficas con cuadrícula blanca
plt.rcParams['figure.figsize'] = (12, 6)  # Tamaño por defecto de figuras
plt.rcParams['font.size'] = 10  # Tamaño de fuente
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas
pd.set_option('display.precision', 4)  # Precisión decimal en display

#Crear directorios para guardar modelos, predicciones y resultados
os.makedirs("models_petro", exist_ok=True)
os.makedirs("predictions_petro", exist_ok=True)
os.makedirs("results_petro", exist_ok=True)

# Importaciones adicionales para deep learning y búsqueda de hiperparámetros
import time
import itertools
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import EfficientNetB0, VGG16, MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Keras Tuner para búsqueda de hiperparámetros (GridSearch)
try:
    import keras_tuner as kt
    KERAS_TUNER_AVAILABLE = True
    print("keras-tuner disponible.")
except ImportError:
    KERAS_TUNER_AVAILABLE = False
    print("keras-tuner no instalado. Instalar con: pip install keras-tuner")
    print("Las celdas de GridSearch usarán busqueda manual como fallback.")

# Reproducibilidad
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow: {tf.__version__}")
print(f"GPUs disponibles: {tf.config.list_physical_devices('GPU')}") 

keras-tuner no instalado. Instalar con: pip install keras-tuner
Las celdas de GridSearch usarán busqueda manual como fallback.
TensorFlow: 2.21.0
GPUs disponibles: []


In [20]:
#Leer archivo con datos originales
df = pd.read_csv("inferencia.csv")

#Copia de dataset
df_clean = df.copy()

logmass_mean = df_clean["LogMass"].mean()
logmass_std = df_clean["LogMass"].std()

def inverse_logmass_scaled(y_scaled):
    return y_scaled * logmass_std + logmass_mean

#REEMPLAZAR PLACEHOLDERS POR NaN
placeholder_map = {
    "C": [-999],
    "A": [-999],
    "S": [-999, -102.97, -118.68],
    "nsa_sersic_mass": [-9999],
    "nsa_sersic_ba": [-9999],
    "nsa_sersic_n": [-9999]
}

for col, values in placeholder_map.items():
    df_clean[col] = df_clean[col].replace(values, np.nan)

#DEFINICIÓN DE VARIABLES
# Variables con asimetría fuerte o potencialmente problemáticas
skewed_cols = [
    "nsa_sersic_mass",
    "PETRO_TH90",
    "modelMag_r",
    "C",
    "A",
    "S"
]

# Variables que ya están bastante estabilizadas o en log
normal_cols = [
    "LogMass",
    "nsa_sersic_ba",
    "nsa_sersic_n",
    "log_age_mean_LW",
    "log_ZH_mean_LW",
    "log_SFR_ssp",
    "log_SFR_Ha",
    "vel_sigma_Re"
]

# Coordenadas
coordinate_cols = [
    "objra",
    "objdec"
]

#Orden original de las columnas
final_columns = [
    "objra",
    "objdec",
    "C",
    "A",
    "S",
    "nsa_sersic_mass",
    "LogMass",
    "nsa_sersic_ba",
    "nsa_sersic_n",
    "PETRO_TH90",
    "log_age_mean_LW",
    "log_ZH_mean_LW",
    "log_SFR_ssp",
    "log_SFR_Ha",
    "vel_sigma_Re",
    "modelMag_r"
]

# Filtrar solo columnas existentes
skewed_cols = [c for c in skewed_cols if c in df_clean.columns]
normal_cols = [c for c in normal_cols if c in df_clean.columns]
coordinate_cols = [c for c in coordinate_cols if c in df_clean.columns]

# Para variables sesgadas: imputación + Yeo-Johnson + escalado
skewed_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("yeojohnson", PowerTransformer(method="yeo-johnson", standardize=False)),
    ("scaler", StandardScaler())
])

# Para variables ya estables: imputación + escalado
normal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Para coordenadas: imputación + escalado
coordinate_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# COLUMN TRANSFORMER
preprocessor = ColumnTransformer([
    ("skewed_vars", skewed_pipeline, skewed_cols),
    ("normal_vars", normal_pipeline, normal_cols),
    ("coordinates", coordinate_pipeline, coordinate_cols)
], remainder="drop")

# AJUSTAR Y TRANSFORMAR
X_transformed = preprocessor.fit_transform(df_clean)

transformer_columns = skewed_cols + normal_cols + coordinate_cols

df_transformed_numeric = pd.DataFrame(
    X_transformed,
    columns=transformer_columns,
    index=df_clean.index
)

# Reordenar columnas al formato deseado
df_transformed_numeric = df_transformed_numeric[final_columns]

# Conservar columna name como identificador de cada galaxia
df_transformed = pd.concat(
    [
        df_clean[["name"]],
        df_transformed_numeric
    ],
    axis=1
)

print("Shape original:", df_clean.shape)
print("Shape transformado:", df_transformed.shape)
df_transformed.head()

Shape original: (10126, 17)
Shape transformado: (10126, 17)


,name,objra,objdec,C,A,S,nsa_sersic_mass,LogMass,nsa_sersic_ba,nsa_sersic_n,PETRO_TH90,log_age_mean_LW,log_ZH_mean_LW,log_SFR_ssp,log_SFR_Ha,vel_sigma_Re,modelMag_r
0,manga-10001-12701,-0.6523,1.6126,-1.3471,-0.0074,-0.0309,-0.9490,-0.9403,-1.2084,-1.2748,-0.9719,-1.1521,-1.3084,0.8322,0.6072,2.0522,0.8076
1,manga-10001-12702,-0.6482,1.6060,-0.6744,-0.3508,-0.2956,-0.6325,-0.6049,-0.4241,-0.9179,0.1545,-1.0850,-0.7010,0.5290,0.1930,2.0572,1.0683
2,manga-10001-12703,-0.6176,1.5844,-0.0944,-0.2080,0.1655,-0.0753,-0.0354,-1.7963,-0.5420,-0.0049,-0.7376,-0.1556,0.7577,0.7587,0.6348,0.1766
3,manga-10001-12704,-0.6442,1.6170,0.1224,0.0694,0.7020,-0.7591,-0.7380,-2.0492,-1.2099,1.4307,-0.7461,-0.7883,-0.0408,0.3075,0.7127,-0.8335
4,manga-10001-12705,-0.6080,1.6044,-0.6729,-0.1478,-0.1510,-0.1074,-0.0674,-0.5909,-1.0158,-0.3170,-1.1710,-1.2657,1.0966,1.0637,1.9645,0.1993


In [21]:
# CONSTRUCCIÓN DEL DATASET SUPERVISADO PARA LogMass
# Relación imagen <-> variable objetivo mediante 'name'

target_col = "LogMass"

df_logmass = df_transformed[["name", target_col]].copy()
df_logmass = df_logmass.dropna(subset=[target_col])

# Ajustar ruta según entorno local
image_dir = Path(r"/Users/gustavomorales/Desktop/ITESM_MNA/Proyecto integrador/Imagenes_petro")

df_logmass["image_path"]   = df_logmass["name"].apply(lambda x: image_dir / f"{x}.jpg")
df_logmass["image_exists"] = df_logmass["image_path"].apply(lambda p: p.exists())

print("Total registros:      ", len(df_logmass))
print("Imagenes encontradas: ", df_logmass["image_exists"].sum())
print("Imagenes faltantes:   ", (~df_logmass["image_exists"]).sum())

df_logmass = df_logmass[df_logmass["image_exists"]].copy()
df_logmass = df_logmass[["name", target_col, "image_path"]].copy()

# TRAIN / VALIDATION / TEST SPLIT
# 70% entrenamiento | 15% validacion | 15% prueba
# Misma semilla que Avance 3 para comparabilidad

train_df, temp_df = train_test_split(df_logmass, test_size=0.30, random_state=42)
val_df,   test_df = train_test_split(temp_df,    test_size=0.50, random_state=42)

print("\nShape dataset LogMass:", df_logmass.shape)
print("Train:     ", train_df.shape)
print("Validation:", val_df.shape)
print("Test:      ", test_df.shape)

Total registros:       10126
Imagenes encontradas:  10001
Imagenes faltantes:    125

Shape dataset LogMass: (10001, 3)
Train:      (7000, 3)
Validation: (1500, 3)
Test:       (1501, 3)


In [22]:
# PARAMETROS GLOBALES Y FUNCIONES DE CARGA DE IMAGENES


ensemble_times = {} #Registrar el tiempo de entrenamiento de los ensambles
IMG_SIZE   = (224, 224)   # Compatible con todos los modelos
BATCH_SIZE = 16           # Consistente con Avance 3

def load_image_regression(image_path, target):
    """
    Carga, decodifica, redimensiona y normaliza una imagen JPEG.

    Parameters
    ----------
    image_path : tf.string
    target     : tf.float32  valor objetivo normalizado (LogMass)

    Returns
    -------
    image  : tf.Tensor shape (224, 224, 3) float32, valores en [0, 1]
    target : tf.float32
    """
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0
    return image, target


def create_dataset(dataframe, shuffle=True):
    """
    Construye un tf.data.Dataset desde un DataFrame con rutas de imagenes.

    Parameters
    ----------
    dataframe : pd.DataFrame  columnas 'image_path' y target_col
    shuffle   : bool

    Returns
    -------
    dataset : tf.data.Dataset
    """
    image_paths = dataframe["image_path"].astype(str).values
    targets     = dataframe[target_col].astype("float32").values

    dataset = tf.data.Dataset.from_tensor_slices((image_paths, targets))
    dataset = dataset.map(load_image_regression, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(dataframe), seed=42)
    dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return dataset


train_ds = create_dataset(train_df, shuffle=True)
val_ds   = create_dataset(val_df,   shuffle=False)
test_ds  = create_dataset(test_df,  shuffle=False)

print("Datasets creados exitosamente.")

Datasets creados exitosamente.


In [23]:
# Heredada del Avance 3; calcula MAE, RMSE, R2, Scatter, Bias

def evaluate_regression_complete(y_true, y_pred, variable_name='LogMass'):
    """
    Evalua un modelo de regresion con metricas astronomicas completas.

    Parameters
    ----------
    y_true        : array-like  valores reales
    y_pred        : array-like  valores predichos
    variable_name : str

    Returns
    -------
    dict : R2, MAE, RMSE, MSE, Scatter, Bias, RMSE_MAE_ratio, Categoria
    """
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()

    mae      = mean_absolute_error(y_true, y_pred)
    mse      = mean_squared_error(y_true, y_pred)
    rmse     = np.sqrt(mse)
    r2       = r2_score(y_true, y_pred)
    residuals = y_true - y_pred
    scatter   = np.std(residuals)
    bias      = np.mean(residuals)
    rmse_mae  = rmse / mae if mae > 0 else np.inf

    print(f"Metricas de Regresion para {variable_name}:")
    print(f"  R2 Score:        {r2:.4f}  ({r2*100:.2f}% varianza explicada)")
    print(f"  MAE:             {mae:.4f} dex")
    print(f"  RMSE:            {rmse:.4f} dex")
    print(f"  MSE:             {mse:.4f} dex2")
    print(f"  Scatter (sigma): {scatter:.4f} dex")
    print(f"  Bias:            {bias:+.4f} dex")
    print(f"  RMSE/MAE ratio:  {rmse_mae:.2f}")
    print()

    mae_f = 10 ** mae; scat_f = 10 ** scatter
    print("Interpretacion fisica (escala lineal):")
    print(f"  Error promedio (MAE):   {mae_f:.2f}x")
    print(f"  Dispersion tipica:      +/-{scat_f:.2f}x")
    print(f"  Rango de error tipico:  [{1/mae_f:.2f}x, {mae_f:.2f}x]")
    if   abs(bias) < 0.05: print("  Sin sesgo sistematico significativo (bias ~ 0)")
    elif bias > 0:         print(f"  Sesgo positivo: modelo subestima ({bias:+.4f} dex)")
    else:                  print(f"  Sesgo negativo: modelo sobreestima ({bias:+.4f} dex)")
    print()

    if   r2 >= 0.90: cat, desc = "EXCELENTE", "Modelo de alta precision"
    elif r2 >= 0.80: cat, desc = "MUY BUENO", "Baseline fuerte"
    elif r2 >= 0.70: cat, desc = "BUENO",     "Desempeno aceptable"
    elif r2 >= 0.50: cat, desc = "REGULAR",   "Mejora requerida"
    else:            cat, desc = "INACEPTABLE","Modelo no viable"

    print(f"Evaluacion cualitativa: {cat} - {desc}")
    print()
    print("Diagnostico de errores:")
    if   rmse_mae > 2.0: print(f"  ADVERTENCIA: RMSE/MAE={rmse_mae:.2f} - outliers severos")
    elif rmse_mae > 1.5: print(f"  NOTA: RMSE/MAE={rmse_mae:.2f} - algunos errores grandes")
    else:                print(f"  RMSE/MAE={rmse_mae:.2f} - distribucion normal, sin outliers extremos")

    return {'R2': r2, 'MAE': mae, 'RMSE': rmse, 'MSE': mse,
            'Scatter': scatter, 'Bias': bias,
            'RMSE_MAE_ratio': rmse_mae, 'Categoria': cat}



# Funciones auxiliares de entrenamiento y visualizacion


def get_predictions(model, dataset):
    """Obtiene predicciones y etiquetas reales de un tf.data.Dataset."""
    y_true_list, y_pred_list = [], []
    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        y_true_list.extend(labels.numpy())
        y_pred_list.extend(preds.flatten())
    return np.array(y_true_list), np.array(y_pred_list)


def train_model(model, train_ds, val_ds, epochs=30, patience=7):
    """
    Entrena un modelo Keras con EarlyStopping y ReduceLROnPlateau.

    Parameters
    ----------
    model    : tf.keras.Model  (ya compilado)
    train_ds, val_ds : tf.data.Dataset
    epochs   : int
    patience : int  paciencia EarlyStopping

    Returns
    -------
    history : tf.keras.callbacks.History
    elapsed : float  segundos totales
    """
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=patience,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, min_lr=1e-7, verbose=1)
    ]
    start   = time.time()
    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=epochs, callbacks=callbacks, verbose=1)
    elapsed = time.time() - start
    print(f"\nTiempo de entrenamiento: {elapsed:.1f}s ({elapsed/60:.1f} min)")
    return history, elapsed


def plot_training_curves(history, model_name):
    """Grafica curvas de perdida (MSE) y MAE de entrenamiento/validacion."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['loss'],     label='Train Loss', lw=2)
    axes[0].plot(history.history['val_loss'], label='Val Loss',   lw=2)
    axes[0].set_title(f'Curva de Perdida (MSE) - {model_name}')
    axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('MSE')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)
    axes[1].plot(history.history['mae'],     label='Train MAE', lw=2)
    axes[1].plot(history.history['val_mae'], label='Val MAE',   lw=2)
    axes[1].set_title(f'Curva de MAE - {model_name}')
    axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('MAE (dex)')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


def plot_combined_history(h1, h2, model_name):
    """
    Grafica curvas de dos fases de entrenamiento (feature extraction + fine-tuning)
    unificadas con linea vertical separadora.
    Retorna el total de epocas ejecutadas.
    """
    sep   = len(h1.history['loss'])
    loss  = h1.history['loss']     + h2.history['loss']
    vloss = h1.history['val_loss'] + h2.history['val_loss']
    mae   = h1.history['mae']      + h2.history['mae']
    vmae  = h1.history['val_mae']  + h2.history['val_mae']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, y, yv, ylabel, title in [
        (axes[0], loss, vloss, 'MSE',       f'Curva de Perdida - {model_name}'),
        (axes[1], mae,  vmae,  'MAE (dex)', f'Curva de MAE - {model_name}')
    ]:
        ax.plot(y,  label='Train', lw=2)
        ax.plot(yv, label='Val',   lw=2)
        ax.axvline(x=sep, color='red', ls='--', alpha=0.7, label='Inicio Fine-Tuning')
        ax.set_title(title); ax.set_xlabel('Epoca'); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    return len(loss)


# Registro global de resultados
results_registry = {}

print("Funciones utilitarias definidas correctamente.")

Funciones utilitarias definidas correctamente.


## Recuperación de modelos individuales

In [24]:
from tensorflow.keras.models import load_model

#Importar modelos y predicciones orginales
model_m0 = load_model("models/M0_CNN_Baseline.keras")
model_m1 = load_model("models/M1_CNN_Tuned.keras")
model_m2 = load_model("models/M2_ResNet_Custom.keras")
model_m3 = load_model("models/M3_EfficientNetB0.keras")

#Importar modelos y predicciones con imagenes con escala diferente (PETRO)
model_m0_petro = load_model("models_petro/M0_CNN_Baseline_petro.keras")
model_m1_petro = load_model("models_petro/M1_CNN_Tuned_petro.keras")
model_m2_petro = load_model("models_petro/M2_ResNet_Custom.keras")
model_m3_petro = load_model("models_petro/M3_EfficientNetB0.keras")

#Importar modelos y predicciones con imagenes con sersic_ba > 0.65
model_m0_sersic_ba = load_model("models_sersic_ba/M0_CNN_Baseline_sersic_ba.keras")
model_m1_sersic_ba = load_model("models_sersic_ba/M1_CNN_Tuned_sersic_ba.keras")
model_m2_sersic_ba = load_model("models_sersic_ba/M2_ResNet_Custom.keras")
model_m3_sersic_ba = load_model("models_sersic_ba/M3_EfficientNetB0.keras")

In [25]:
#Predicciones de modelos petro

y_val_m0, y_pred_val_m0 = get_predictions(model_m0_petro, val_ds)
y_val_m1, y_pred_val_m1 = get_predictions(model_m1_petro, val_ds)
y_val_m2, y_pred_val_m2 = get_predictions(model_m2_petro, val_ds)
y_val_m3, y_pred_val_m3 = get_predictions(model_m3_petro, val_ds)

y_test_m0, y_pred_test_m0 = get_predictions(model_m0_petro, test_ds)
y_test_m1, y_pred_test_m1 = get_predictions(model_m1_petro, test_ds)
y_test_m2, y_pred_test_m2 = get_predictions(model_m2_petro, test_ds)
y_test_m3, y_pred_test_m3 = get_predictions(model_m3_petro, test_ds)

# VERIFICAR CONSISTENCIA DE TARGETS

print("Validación:")
print("M0 vs M1:", np.allclose(y_val_m0, y_val_m1))
print("M0 vs M2:", np.allclose(y_val_m0, y_val_m2))
print("M0 vs M3:", np.allclose(y_val_m0, y_val_m3))

print("\nTest:")
print("M0 vs M1:", np.allclose(y_test_m0, y_test_m1))
print("M0 vs M2:", np.allclose(y_test_m0, y_test_m2))
print("M0 vs M3:", np.allclose(y_test_m0, y_test_m3))

Validación:
M0 vs M1: True
M0 vs M2: True
M0 vs M3: True

Test:
M0 vs M1: True
M0 vs M2: True
M0 vs M3: True


In [26]:
y_val = y_val_m0
y_test = y_test_m0

In [27]:
#Guardar predicciones

os.makedirs("predictions_avance5_petro", exist_ok=True)

np.save("predictions_avance5_petro/y_val.npy", y_val)
np.save("predictions_avance5_petro/y_test.npy", y_test)

np.save("predictions_avance5_petro/M0_y_pred_val.npy", y_pred_val_m0)
np.save("predictions_avance5_petro/M1_y_pred_val.npy", y_pred_val_m1)
np.save("predictions_avance5_petro/M2_y_pred_val.npy", y_pred_val_m2)
np.save("predictions_avance5_petro/M3_y_pred_val.npy", y_pred_val_m3)

np.save("predictions_avance5_petro/M0_y_pred_test.npy", y_pred_test_m0)
np.save("predictions_avance5_petro/M1_y_pred_test.npy", y_pred_test_m1)
np.save("predictions_avance5_petro/M2_y_pred_test.npy", y_pred_test_m2)
np.save("predictions_avance5_petro/M3_y_pred_test.npy", y_pred_test_m3)

___
# Ensamble 1: Simple Average (Heterogéneo)

Este ensamble heterogéneo combina las predicciones del modelo CNN Baseline (M0) y del modelo ResNet Custom (M2). Se considera heterogéneo debido a que integra arquitecturas de aprendizaje profundo diferentes, las cuales pueden capturar características complementarias de las imágenes astronómicas.

La predicción final se obtiene mediante el promedio aritmético de las predicciones generadas por ambos modelos.

El objetivo de este enfoque es reducir la varianza y compensar errores individuales de cada modelo, aprovechando la diversidad arquitectónica para mejorar la capacidad de generalización.

In [28]:
# ENSAMBLE 1: SIMPLE AVERAGE HETEROGÉNEO
# CNN Baseline (M0) + ResNet Custom (M2)

start_time = time.perf_counter()

y_pred_test_avg_het = (
    y_pred_test_m0 +
    y_pred_test_m2
) / 2

y_pred_val_avg_het = (
    y_pred_val_m0 +
    y_pred_val_m2
) / 2

time_e1_het = time.perf_counter() - start_time

print(f"Tiempo E1 heterogéneo: {time_e1_het:.6f} s")

Tiempo E1 heterogéneo: 0.000043 s


In [29]:
print("ENSAMBLE 1 - SIMPLE AVERAGE HETEROGÉNEO\n")

metrics_avg_het = evaluate_regression_complete(
    y_test,
    y_pred_test_avg_het,
    "LogMass"
)

results_registry["E1_Het_SimpleAverage_Het"] = metrics_avg_het

ENSAMBLE 1 - SIMPLE AVERAGE HETEROGÉNEO

Metricas de Regresion para LogMass:
  R2 Score:        0.7782  (77.82% varianza explicada)
  MAE:             0.3581 dex
  RMSE:            0.4739 dex
  MSE:             0.2246 dex2
  Scatter (sigma): 0.4723 dex
  Bias:            +0.0385 dex
  RMSE/MAE ratio:  1.32

Interpretacion fisica (escala lineal):
  Error promedio (MAE):   2.28x
  Dispersion tipica:      +/-2.97x
  Rango de error tipico:  [0.44x, 2.28x]
  Sin sesgo sistematico significativo (bias ~ 0)

Evaluacion cualitativa: BUENO - Desempeno aceptable

Diagnostico de errores:
  RMSE/MAE=1.32 - distribucion normal, sin outliers extremos


In [30]:
np.save(
    "predictions_avance5_petro/E1_Het_y_pred_test.npy",
    y_pred_test_avg_het
)

np.save(
    "predictions_avance5_petro/E1_Het_y_pred_val.npy",
    y_pred_val_avg_het
)

# Ensamble 1: Simple Average (Homogéneo)

Este ensamble homogéneo combina las predicciones del modelo CNN Baseline (M0) y del modelo CNN Tuned (M1). Se considera homogéneo debido a que ambos modelos pertenecen a la misma familia de arquitecturas convolucionales (CNN) y comparten principios de aprendizaje similares, aunque difieren en la configuración de hiperparámetros utilizada durante el entrenamiento.

In [31]:
# ENSAMBLE 1: SIMPLE AVERAGE HOMOGÉNEO
# CNN Baseline (M0) + CNN Tuned (M1)

start_time = time.perf_counter()

y_pred_test_avg_hom = (
    y_pred_test_m0 +
    y_pred_test_m1
) / 2

y_pred_val_avg_hom = (
    y_pred_val_m0 +
    y_pred_val_m1
) / 2

time_e1_hom = time.perf_counter() - start_time

print(f"Tiempo E1 homogéneo: {time_e1_hom:.6f} s")

Tiempo E1 homogéneo: 0.000052 s


In [32]:
print("ENSAMBLE 1 - SIMPLE AVERAGE HOMOGÉNEO\n")

metrics_avg_hom = evaluate_regression_complete(
    y_test,
    y_pred_test_avg_hom,
    "LogMass"
)

results_registry["E1_SimpleAverage_Hom"] = metrics_avg_hom

ENSAMBLE 1 - SIMPLE AVERAGE HOMOGÉNEO

Metricas de Regresion para LogMass:
  R2 Score:        0.7511  (75.11% varianza explicada)
  MAE:             0.3843 dex
  RMSE:            0.5021 dex
  MSE:             0.2521 dex2
  Scatter (sigma): 0.4952 dex
  Bias:            +0.0828 dex
  RMSE/MAE ratio:  1.31

Interpretacion fisica (escala lineal):
  Error promedio (MAE):   2.42x
  Dispersion tipica:      +/-3.13x
  Rango de error tipico:  [0.41x, 2.42x]
  Sesgo positivo: modelo subestima (+0.0828 dex)

Evaluacion cualitativa: BUENO - Desempeno aceptable

Diagnostico de errores:
  RMSE/MAE=1.31 - distribucion normal, sin outliers extremos


In [33]:
np.save(
    "predictions_avance5_petro/E1_Hom_y_pred_test.npy",
    y_pred_test_avg_hom
)

np.save(
    "predictions_avance5_petro/E1_Hom_y_pred_val.npy",
    y_pred_val_avg_hom
)

___
# Ensamble 2: Weighted Average (Heterogéneo)

Este ensamble heterogéneo combina las predicciones de los modelos CNN Baseline (M0), CNN Tuned (M1) y ResNet Custom (M2). A diferencia del promedio simple, cada modelo contribuye de manera distinta a la predicción final mediante pesos previamente definidos en función de su desempeño individual.

Este método busca otorgar mayor influencia a los modelos con mejor desempeño, manteniendo al mismo tiempo la diversidad de arquitecturas para mejorar la precisión global del sistema.

In [34]:
#Busqueda de los mejores pesos para cada modelo
from itertools import product
from sklearn.metrics import r2_score

results_weighted = []

for w0, w1 in product(
    [0.1, 0.2, 0.3, 0.4],
    [0.1, 0.2, 0.3, 0.4]
):
    w2 = 1 - w0 - w1

    if w2 <= 0:
        continue

    y_pred = (
        w0 * y_pred_val_m0 +
        w1 * y_pred_val_m1 +
        w2 * y_pred_val_m2
    )

    results_weighted.append({
        "w0": w0,
        "w1": w1,
        "w2": w2,
        "R2": r2_score(y_val, y_pred)
    })

pd.DataFrame(results_weighted)\
    .sort_values("R2", ascending=False)\
    .head(10)

,w0,w1,w2,R2
8,0.3,0.1,0.6,0.7613
9,0.3,0.2,0.5,0.7604
5,0.2,0.2,0.6,0.7599
12,0.4,0.1,0.5,0.7598
4,0.2,0.1,0.7,0.7592
6,0.2,0.3,0.5,0.7585
10,0.3,0.3,0.4,0.7575
13,0.4,0.2,0.4,0.7575
2,0.1,0.3,0.6,0.7560
1,0.1,0.2,0.7,0.7558


In [35]:
# ENSAMBLE 2: WEIGHTED AVERAGE HETEROGÉNEO
# M0 + M1 + M2

start_time = time.perf_counter()

w0 = 0.30
w1 = 0.20
w2 = 0.50

y_pred_test_weighted = (
    w0 * y_pred_test_m0 +
    w1 * y_pred_test_m1 +
    w2 * y_pred_test_m2
)

y_pred_val_weighted = (
    w0 * y_pred_val_m0 +
    w1 * y_pred_val_m1 +
    w2 * y_pred_val_m2
)

metrics_weighted = evaluate_regression_complete(
    y_test,
    y_pred_test_weighted,
    "LogMass"
)

time_e2 = time.perf_counter() - start_time

results_registry["E2_WeightedAverage"] = metrics_weighted

print(f"Tiempo E2 Weighted Average: {time_e2:.6f} s")

Metricas de Regresion para LogMass:
  R2 Score:        0.7817  (78.17% varianza explicada)
  MAE:             0.3540 dex
  RMSE:            0.4702 dex
  MSE:             0.2211 dex2
  Scatter (sigma): 0.4701 dex
  Bias:            +0.0081 dex
  RMSE/MAE ratio:  1.33

Interpretacion fisica (escala lineal):
  Error promedio (MAE):   2.26x
  Dispersion tipica:      +/-2.95x
  Rango de error tipico:  [0.44x, 2.26x]
  Sin sesgo sistematico significativo (bias ~ 0)

Evaluacion cualitativa: BUENO - Desempeno aceptable

Diagnostico de errores:
  RMSE/MAE=1.33 - distribucion normal, sin outliers extremos
Tiempo E2 Weighted Average: 0.000734 s


In [36]:
np.save(
    "predictions_avance5_petro/E2_y_pred_test.npy",
    y_pred_test_weighted
)

np.save(
    "predictions_avance5_petro/E2_y_pred_val.npy",
    y_pred_val_weighted
)

---

# Ensamble 3: Stacking Linear Regression (Heterogéneo)

Este ensamble heterogéneo utiliza las predicciones de los modelos CNN Baseline (M0), CNN Tuned (M1), ResNet Custom (M2) y EfficientNetB0 (M3) como variables de entrada para entrenar un meta-modelo basado en regresión lineal.

A diferencia de los ensambles por promedio, el meta-modelo aprende automáticamente la contribución óptima de cada modelo base a partir de los datos de validación.

El objetivo es aprovechar las fortalezas individuales de cada arquitectura y aprender una combinación más efectiva que un promedio simple o ponderado definido manualmente.

In [37]:
# ENSAMBLE 3: STACKING LINEAR REGRESSION
# Este ensamble utiliza las predicciones de varios modelos base
# como nuevas variables de entrada para entrenar un meta-modelo.

from sklearn.linear_model import LinearRegression

start_time = time.perf_counter()

# Construir matriz de entrada para VALIDACIÓN
# Cada columna representa las predicciones de un modelo base

X_meta_val = np.column_stack([
    y_pred_val_m0,
    y_pred_val_m1,
    y_pred_val_m2
])

# Variable objetivo para entrenar el meta-modelo
y_meta_val = y_val

# Crear meta-modelo
meta_lr = LinearRegression()

# Entrenar meta-modelo
meta_lr.fit(
    X_meta_val,
    y_meta_val
)

# Coeficientes aprendidos
for i, coef in enumerate(meta_lr.coef_):
    print(f"Modelo {i}: {coef:.4f}")

print("Intercept:", meta_lr.intercept_)

# Construir matriz para TEST
X_meta_test = np.column_stack([
    y_pred_test_m0,
    y_pred_test_m1,
    y_pred_test_m2
])

# Predicción final del ensamble
y_pred_test_stack_lr = meta_lr.predict(
    X_meta_test
)

time_e3 = time.perf_counter() - start_time

print(f"Tiempo E3 Stacking LR: {time_e3:.6f} s")

Modelo 0: 0.3199
Modelo 1: 0.1534
Modelo 2: 0.5707
Intercept: -0.007218279
Tiempo E3 Stacking LR: 0.000989 s


In [38]:
print("ENSAMBLE 3 - STACKING LINEAR REGRESSION\n")

metrics_stack_lr = evaluate_regression_complete(
    y_test,
    y_pred_test_stack_lr,
    "LogMass"
)

results_registry["E3_Stacking_LR"] = metrics_stack_lr

ENSAMBLE 3 - STACKING LINEAR REGRESSION

Metricas de Regresion para LogMass:
  R2 Score:        0.7815  (78.15% varianza explicada)
  MAE:             0.3523 dex
  RMSE:            0.4704 dex
  MSE:             0.2212 dex2
  Scatter (sigma): 0.4702 dex
  Bias:            +0.0114 dex
  RMSE/MAE ratio:  1.34

Interpretacion fisica (escala lineal):
  Error promedio (MAE):   2.25x
  Dispersion tipica:      +/-2.95x
  Rango de error tipico:  [0.44x, 2.25x]
  Sin sesgo sistematico significativo (bias ~ 0)

Evaluacion cualitativa: BUENO - Desempeno aceptable

Diagnostico de errores:
  RMSE/MAE=1.34 - distribucion normal, sin outliers extremos


In [39]:
np.save(
    "predictions_avance5_petro/E3_y_pred_test.npy",
    y_pred_test_stack_lr
)

---

# Ensamble 4: Stacking Ridge Regression (Heterogéneo)

Este ensamble heterogéneo utiliza las predicciones de los modelos CNN Baseline (M0), CNN Tuned (M1) y ResNet Custom (M2) como variables de entrada para un meta-modelo Ridge Regression.

A diferencia de la regresión lineal simple, Ridge incorpora regularización L2, lo cual ayuda a controlar coeficientes excesivamente grandes cuando las predicciones de los modelos base están correlacionadas. Esto es relevante porque M0 y M1 pertenecen a la misma familia CNN y pueden generar salidas similares.

El objetivo es obtener una combinación más estable de los modelos base, reduciendo el riesgo de sobreajuste del meta-modelo.

In [40]:
# ENSAMBLE 4: STACKING CON RIDGE REGRESSION

from sklearn.linear_model import Ridge

#Construir matriz de entrada para el meta-modelo
X_meta_val = np.column_stack([
    y_pred_val_m0,  # CNN Baseline
    y_pred_val_m1,  # CNN Tuned
    y_pred_val_m2   # ResNet Custom
])

y_meta_val = y_val

# Buscando el mejor valor alpha
alphas = [0.001, 0.01, 0.1, 1.0, 10.0]

ridge_results = []

for alpha in alphas:
    ridge_model = Ridge(alpha=alpha)

    ridge_model.fit(X_meta_val, y_meta_val)

    y_pred_val_ridge = ridge_model.predict(X_meta_val)

    ridge_results.append({
        "alpha": alpha,
        "R2_val": r2_score(y_meta_val, y_pred_val_ridge),
        "coef_M0": ridge_model.coef_[0],
        "coef_M1": ridge_model.coef_[1],
        "coef_M2": ridge_model.coef_[2],
        "intercept": ridge_model.intercept_
    })

ridge_results_df = pd.DataFrame(ridge_results)
ridge_results_df.sort_values("R2_val", ascending=False)

,alpha,R2_val,coef_M0,coef_M1,coef_M2,intercept
0,0.001,0.7625,0.3199,0.1534,0.5707,-0.0072
1,0.010,0.7625,0.3199,0.1534,0.5707,-0.0072
2,0.100,0.7625,0.3200,0.1537,0.5704,-0.0072
3,1.000,0.7625,0.3201,0.1563,0.5678,-0.0069
4,10.000,0.7625,0.3208,0.1785,0.5450,-0.0048


In [41]:
#Seleccionar mejor alpha según R2 en validación

best_alpha = ridge_results_df.sort_values(
    "R2_val",
    ascending=False
).iloc[0]["alpha"]

print(f"Mejor alpha para Ridge: {best_alpha}")

Mejor alpha para Ridge: 0.001


In [42]:
start_time = time.perf_counter()

meta_ridge = Ridge(alpha=best_alpha)

meta_ridge.fit(
    X_meta_val,
    y_meta_val
)

for i, coef in enumerate(meta_ridge.coef_):
    print(f"Modelo {i}: {coef:.4f}")

print("Intercept:", meta_ridge.intercept_)

y_pred_test_stack_ridge = meta_ridge.predict(X_meta_test)
y_pred_val_stack_ridge = meta_ridge.predict(X_meta_val)

time_e4 = time.perf_counter() - start_time

print(f"Tiempo E4 Stacking Ridge: {time_e4:.6f} s")

Modelo 0: 0.3199
Modelo 1: 0.1534
Modelo 2: 0.5707
Intercept: -0.007219143
Tiempo E4 Stacking Ridge: 0.000855 s


In [43]:
# Entrada para test

X_meta_test = np.column_stack([
    y_pred_test_m0,
    y_pred_test_m1,
    y_pred_test_m2
])

# Predicción final del ensamble Ridge

y_pred_test_stack_ridge = meta_ridge.predict(X_meta_test)

In [44]:
# 7. Evaluar ensamble

print("ENSAMBLE 4 - STACKING RIDGE REGRESSION\n")

metrics_stack_ridge = evaluate_regression_complete(
    y_test,
    y_pred_test_stack_ridge,
    "LogMass"
)

results_registry["E4_Stacking_Ridge"] = metrics_stack_ridge

ENSAMBLE 4 - STACKING RIDGE REGRESSION

Metricas de Regresion para LogMass:
  R2 Score:        0.7815  (78.15% varianza explicada)
  MAE:             0.3523 dex
  RMSE:            0.4704 dex
  MSE:             0.2212 dex2
  Scatter (sigma): 0.4702 dex
  Bias:            +0.0114 dex
  RMSE/MAE ratio:  1.34

Interpretacion fisica (escala lineal):
  Error promedio (MAE):   2.25x
  Dispersion tipica:      +/-2.95x
  Rango de error tipico:  [0.44x, 2.25x]
  Sin sesgo sistematico significativo (bias ~ 0)

Evaluacion cualitativa: BUENO - Desempeno aceptable

Diagnostico de errores:
  RMSE/MAE=1.34 - distribucion normal, sin outliers extremos


In [45]:
np.save(
    "predictions_avance5_petro/E4_y_pred_test.npy",
    y_pred_test_stack_ridge
)

np.save(
    "predictions_avance5_petro/E4_y_pred_val.npy",
    y_pred_val_stack_ridge
)

___

# Comparación de modelos
En esta sección se comparan los modelos individuales desarrollados en la fase anterior con los distintos ensambles construidos en esta actividad. La comparación se realiza utilizando como métrica principal el coeficiente de determinación (R²), complementado con métricas de error como MAE, RMSE y Bias, las cuales permiten evaluar la precisión y estabilidad de las predicciones obtenidas para la variable objetivo LogMass.

Los tiempos de entrenamiento de los modelos individuales fueron obtenidos durante la fase de construcción y optimización de cada arquitectura. En contraste, los tiempos reportados para los ensambles corresponden únicamente al entrenamiento del meta-modelo o al cálculo de la combinación de predicciones, por lo que representan un costo computacional adicional mínimo respecto al entrenamiento de las redes neuronales base.

In [46]:
#Evaluación de modelos indivduales
individual_models = {
    "M0_CNN_Baseline_Petro": model_m0_petro,
    "M1_CNN_Tuned_Petro": model_m1_petro,
    "M2_ResNet_Custom_Petro": model_m2_petro,
    "M3_EfficientNetB0_Petro": model_m3_petro
}

for model_name, model in individual_models.items():
    print(f"\nEvaluando {model_name}\n")

    y_test_ind, y_pred_ind = get_predictions(model, test_ds)

    metrics_ind = evaluate_regression_complete(
        y_test_ind,
        y_pred_ind,
        "LogMass"
    )

    results_registry[model_name] = metrics_ind
results_registry.pop("E1_SimpleAverage", None)


Evaluando M0_CNN_Baseline_Petro

Metricas de Regresion para LogMass:
  R2 Score:        0.7146  (71.46% varianza explicada)
  MAE:             0.4158 dex
  RMSE:            0.5376 dex
  MSE:             0.2890 dex2
  Scatter (sigma): 0.5136 dex
  Bias:            +0.1586 dex
  RMSE/MAE ratio:  1.29

Interpretacion fisica (escala lineal):
  Error promedio (MAE):   2.60x
  Dispersion tipica:      +/-3.26x
  Rango de error tipico:  [0.38x, 2.60x]
  Sesgo positivo: modelo subestima (+0.1586 dex)

Evaluacion cualitativa: BUENO - Desempeno aceptable

Diagnostico de errores:
  RMSE/MAE=1.29 - distribucion normal, sin outliers extremos

Evaluando M1_CNN_Tuned_Petro

Metricas de Regresion para LogMass:
  R2 Score:        0.7100  (71.00% varianza explicada)
  MAE:             0.3989 dex
  RMSE:            0.5419 dex
  MSE:             0.2937 dex2
  Scatter (sigma): 0.5419 dex
  Bias:            +0.0069 dex
  RMSE/MAE ratio:  1.36

Interpretacion fisica (escala lineal):
  Error promedio (MAE):  

In [47]:
# Creación de tabla comparativa

train_times = {
    "M0_CNN_Baseline_Petro": 498.0,
    "M1_CNN_Tuned_Petro": 1285.7,
    "M2_ResNet_Custom_Petro": 1508.2,
    "M3_EfficientNetB0_Petro": 331.0,

    "E1_SimpleAverage_Hom": time_e1_hom,
    "E1_Het_SimpleAverage_Het": time_e1_het,
    "E2_WeightedAverage": time_e2,
    "E3_Stacking_LR": time_e3,
    "E4_Stacking_Ridge": time_e4,
}

model_info = {
    "M0_CNN_Baseline_Petro": ["Individual", "CNN Baseline"],
    "M1_CNN_Tuned_Petro": ["Individual", "CNN Tuned"],
    "M2_ResNet_Custom_Petro": ["Individual", "ResNet Custom"],
    "M3_EfficientNetB0_Petro": ["Individual", "EfficientNetB0"],

    "E1_SimpleAverage_Hom": ["Ensamble", "Simple Average Homogéneo"],
    "E1_Het_SimpleAverage_Het": ["Ensamble", "Simple Average Heterogéneo"],
    "E2_WeightedAverage": ["Ensamble", "Weighted Average Heterogéneo"],
    "E3_Stacking_LR": ["Ensamble", "Stacking Linear Regression"],
    "E4_Stacking_Ridge": ["Ensamble", "Stacking Ridge Regression"],
}

comparison_df = pd.DataFrame(results_registry).T.copy()

comparison_df["Tipo"] = comparison_df.index.map(lambda x: model_info[x][0])
comparison_df["Estrategia"] = comparison_df.index.map(lambda x: model_info[x][1])
comparison_df["Train_Time_s"] = comparison_df.index.map(train_times)
comparison_df["Train_Time_min"] = comparison_df["Train_Time_s"] / 60

cols = [
    "Tipo",
    "Estrategia",
    "R2",
    "MAE",
    "RMSE",
    "Bias",
    "Scatter",
    "Train_Time_s",
    "Train_Time_min"
]

comparison_df = comparison_df[cols].sort_values(
    by="R2",
    ascending=False
)

comparison_df.round(4)

,Tipo,Estrategia,R2,MAE,RMSE,Bias,Scatter,Train_Time_s,Train_Time_min
E2_WeightedAverage,Ensamble,Weighted Average Heterogéneo,0.7817,0.354,0.4702,0.0081,0.4701,0.0007,0.0000
E3_Stacking_LR,Ensamble,Stacking Linear Regression,0.7815,0.3523,0.4704,0.0114,0.4702,0.0010,0.0000
E4_Stacking_Ridge,Ensamble,Stacking Ridge Regression,0.7815,0.3523,0.4704,0.0114,0.4702,0.0009,0.0000
E1_Het_SimpleAverage_Het,Ensamble,Simple Average Heterogéneo,0.7782,0.3581,0.4739,0.0385,0.4723,0.0000,0.0000
M2_ResNet_Custom_Petro,Individual,ResNet Custom,0.7517,0.378,0.5015,-0.0817,0.4947,1508.2000,25.1367
E1_SimpleAverage_Hom,Ensamble,Simple Average Homogéneo,0.7511,0.3843,0.5021,0.0828,0.4952,0.0001,0.0000
M0_CNN_Baseline_Petro,Individual,CNN Baseline,0.7146,0.4158,0.5376,0.1586,0.5136,498.0000,8.3000
M1_CNN_Tuned_Petro,Individual,CNN Tuned,0.71,0.3989,0.5419,0.0069,0.5419,1285.7000,21.4283
M3_EfficientNetB0_Petro,Individual,EfficientNetB0,-0.0007,0.8535,1.0067,0.0441,1.0057,331.0000,5.5167


Los resultados obtenidos muestran que las estrategias de ensamble superaron consistentemente a los modelos individuales evaluados. Mientras que el mejor modelo individual fue M2 ResNet Custom, con un coeficiente de determinación de 0.7517 y un RMSE de 0.5015, todos los ensambles heterogéneos lograron mejoras adicionales tanto en capacidad explicativa como en reducción de error.

Los mejores resultados fueron obtenidos por los ensambles basados en Stacking, alcanzando un valor de R² = 0.7815, lo que representa una mejora aproximada de tres puntos porcentuales respecto al mejor modelo individual.

Es relevante mencionar que los ensambles alcanzan estas mejoras con un costo computacional adicional prácticamente despreciable. Mientras que los modelos individuales requirieron entre 5 y 25 minutos de entrenamiento, los ensambles basados en combinación de predicciones y meta-modelos lineales fueron construidos en fracciones de segundo. Esto demuestra que las estrategias de ensamble representan una alternativa eficiente para incrementar el desempeño predictivo sin necesidad de entrenar arquitecturas más complejas o costosas computacionalmente.

___

# Selección del modelo final

De acuerdo con los resultados obtenidos, los mejores desempeños fueron alcanzados por los ensambles heterogéneos. Particularmente, los modelos E2 (Weighted Average), E3 (Stacking Linear Regression) y E4 (Stacking Ridge Regression) obtuvieron resultados muy similares en todas las métricas evaluadas, superando de manera consistente a los modelos individuales.

E2 Weighted Average Heterogéneo alcanzó el mayor R² = 0.7817, así como el menor RMSE (0.4702) entre todos los modelos evaluados.

El modelo E2 presenta ventajas importantes en términos de simplicidad e interpretabilidad. A diferencia de los enfoques de Stacking, el ensamble ponderado no requiere el entrenamiento de un meta-modelo adicional, sino que combina directamente las predicciones de los modelos base mediante pesos previamente determinados. Esto facilita su implementación, reproducibilidad y análisis posterior.

Por estas razones, se selecciona el modelo E2 Weighted Average Heterogéneo como modelo final del proyecto. Este ensamble combina las predicciones de los modelos CNN Baseline, CNN Tuned y ResNet Custom mediante los pesos 0.30, 0.20 y 0.50 respectivamente, logrando explicar aproximadamente el 78% de la variabilidad observada en la masa estelar de las galaxias y representando el mejor equilibrio entre desempeño predictivo y complejidad del modelo.

___

# Análisis del rendimiento con gráficos

___

# Conclusiones

In [ ]:
#Visualizar resultados
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

for i in range(1):

    # SELECCIONAR UNA MUESTRA ALEATORIA
    sample_row = test_df.sample(1).iloc[0]
    img_path = "/Users/gustavomorales/Downloads/leia.jpg"

    # Valor real ESCALADO
    real_logmass_scaled = sample_row[target_col]

    # Convertir a LogMass REAL
    real_logmass = inverse_logmass_scaled(real_logmass_scaled)

    # CARGAR Y PREPROCESAR IMAGEN
    img = image.load_img(
        img_path,
        target_size=IMG_SIZE
    )

    img_array = image.img_to_array(img)
    img_array = img_array / 255.0

    # Agregar dimensión batch
    img_input = np.expand_dims(img_array, axis=0)

    # PREDICCIÓN DEL MODELO (ESCALADA)
    pred_m0_scaled = model_m0_petro.predict(
        img_input,
        verbose=0
    )[0][0]

    pred_m2_scaled = model_m2_petro.predict(
        img_input,
        verbose=0
    )[0][0]

    pred_logmass_scaled = (
        pred_m0_scaled +
        pred_m2_scaled
    ) / 2

    # Convertir predicción a escala real
    pred_logmass = inverse_logmass_scaled(pred_logmass_scaled)

    # Error físico
    error_dex = real_logmass - pred_logmass

    # VISUALIZACIÓN
    plt.figure(figsize=(6,6))
    plt.imshow(img_array)

    plt.title(
        f"Real LogMass: {real_logmass:.3f}\n"
        f"Predicted LogMass: {pred_logmass:.3f}\n"
        f"Residual: {error_dex:+.3f} dex"
    )

    plt.axis("off")
    plt.show()

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x147490220>